In [ ]:

import numpy as np
import pandas as pd
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, RepeatVector, Dense
from sklearn.preprocessing import StandardScaler

# Load and preprocess data
df = pd.read_csv("LP IV Updated Datasets/AutoEncoder/NASA Turbofan ENgine/NASA TurboFan Engine.csv")
features = df.filter(like="sensor").columns

train = df[(df.unit <= 80) & (df.Fault_Label == 0)]
test = df[df.unit > 80]

scaler = StandardScaler().fit(train[features])

datasets = []
for data in [train, test]:
    windows = []
    for _, engine in data.groupby("unit"):
        a = scaler.transform(engine.sort_values("cycle")[features])
        windows.extend([a[i:i+20] for i in range(len(a)-19)])
    datasets.append(np.array(windows))

x_train, x_test = datasets

# Create LSTM Autoencoder
model = Sequential([
    Input(shape=(20, 21)),
    LSTM(16),
    RepeatVector(20),
    LSTM(16, return_sequences=True),
    Dense(21)
])

# Compile
model.compile(
    optimizer='adam',
    loss='mse'
)

# Train
model.fit(x_train, x_train, epochs=10, batch_size=128)

# Detect faults
error = np.mean((x_train - model.predict(x_train)) ** 2, axis=(1, 2))
threshold = np.percentile(error, 95)

error = np.mean((x_test - model.predict(x_test)) ** 2, axis=(1, 2))
faults = error > threshold

print("Faults:", faults.sum())
